# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)
df = pd.read_csv('/Users/sujith/Documents/Data_Visualization/data-viz-class-material/data/gapminder.csv')
df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2000 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [15]:
# Task 1
# YOUR CODE HERE

# Task 1 — Scatter: life expectancy change over time

import plotly.express as px

continent = 'Asia'

df_filtered = df[
    (df['continent'] == continent) &
    (df['year'].isin([2002, 2007]))  # ✅ 2002 instead of 2000
].copy()

df_filtered['gdp_shifted'] = df_filtered['gdpPercap']
df_filtered.loc[df_filtered['year'] == 2007, 'gdp_shifted'] *= 1.05

fig = px.scatter(
    df_filtered,
    x='gdp_shifted',
    y='lifeExp',
    color='year',
    hover_name='country',
    log_x=True,
    color_discrete_map={
        2002: '#888888',   # ✅ updated key
        2007: '#1f77b4'
    },
    labels={
        'gdp_shifted': 'GDP per Capita (log scale)',
        'lifeExp': 'Life Expectancy',
        'year': ''
    }
)

fig.update_traces(marker=dict(size=9, opacity=0.75))

fig.update_layout(
    title="Asian countries show upward and rightward movement from 2002 to 2007",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12)
)

fig.update_xaxes(showgrid=True, gridcolor='#EEEEEE')
fig.update_yaxes(showgrid=True, gridcolor='#EEEEEE')

fig.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [12]:
# Task 2
# YOUR CODE HERE


import math

highlight = 'Asia'

color_map = {
    c: ('#E63946' if c == highlight else '#DDDDDD')
    for c in df['continent'].unique()
}

df_2007 = df[df['year'] == 2007]

fig = px.scatter(
    df_2007,
    x='gdpPercap',
    y='lifeExp',
    size='pop',                 # THIS is correct
    color='continent',
    color_discrete_map=color_map,
    hover_name='country',
    log_x=True,
    size_max=50,                # IMPORTANT: controls scaling
    opacity=0.7,
    labels={
        'gdpPercap': 'GDP per Capita (log scale)',
        'lifeExp': 'Life Expectancy',
        'pop': 'Population',
        'continent': ''
    }
)

# Remove legend
fig.update_traces(showlegend=False)

# Make highlighted continent clearer
fig.update_traces(
    selector=dict(name=highlight),
    marker=dict(line=dict(width=1, color='black'), opacity=0.85)
)

# Add annotation
row = df_2007[df_2007['country'] == 'India'].iloc[0]

fig.add_annotation(
    x=math.log10(row['gdpPercap']),
    y=row['lifeExp'],
    text="India: large population,\nlower income level",
    showarrow=True,
    arrowhead=1,
    ax=40,
    ay=40,
    font=dict(size=11, color='#E63946')
)

fig.update_layout(
    title=dict(
        text="Asia stands out with massive populations<br>and uneven progress in wealth and health",
        x=0.5   # center align
    )
)

fig.update_xaxes(showgrid=True, gridcolor='#EEEEEE')
fig.update_yaxes(showgrid=True, gridcolor='#EEEEEE')

fig.show()